# MobilityAPI — MF Stream Tutorial (Continuous Queries)

This notebook walks through the **streaming** half of MobilityAPI: the
[OGC API – Moving Features – Part 4 (Stream Extension)](https://www.opengis.net/spec/ogcapi-movingfeatures-4/1.0)
continuous-query endpoints, served by the same Go tier over **MobilityDB/MEOS**.

The idea is the **lifting principle**: a temporal float `tfloat` is a function
`t → float`, so any scalar float operation (`ln`, `exp`, `×`, `+`, …) *lifts* to it
by applying pointwise. A **continuous query** applies a lifted operation to a
streaming `tfloat` and delivers the transformed instants — the streaming
counterpart of applying `ln` or `×` to a scalar.

**Engine seam.** The tier runs continuous queries behind a `StreamEngine` seam,
the streaming analogue of the `Backend` seam that abstracts PostgreSQL / DuckDB /
Spark for the request–response API. This notebook is plain HTTP + Server-Sent
Events, so it is **engine-agnostic**: it runs today on the in-process
`meos-local` engine, and the *same notebook* targets the Flink / Kafka / Spark
engines unchanged once they are selected — the seam hides the engine.


## Prerequisites

- The Go tier built **with the streaming engine** and reachable on
  `http://localhost:8088`:
  ```
  CGO_ENABLED=1 go build -tags meos -o mfapi .
  MFAPI_DSN=<dsn> LD_LIBRARY_PATH=/usr/local/lib ./mfapi
  ```
  The default (cgo-free) build serves every other endpoint but reports the
  streaming engine as not built. Point the notebook elsewhere with `MFAPI_HOST`.
- A Jupyter kernel (the first cell installs `requests`).


In [ ]:
%pip install -q requests


In [ ]:
import os, json, requests
HOST = os.environ.get('MFAPI_HOST', 'http://localhost:8088')
S = requests.Session()
def show(r):
    print(r.status_code, r.request.method, r.url.replace(HOST,''))
    try: print(json.dumps(r.json(), indent=2)[:1200])
    except Exception: print(r.text[:600])
    return r
print('tier:', HOST, '->', S.get(HOST + '/health').json())


## 1. A moving feature with a temporal property

We create a small self-contained collection, one moving feature with a short
trajectory, and a `speed` temporal float (`TReal`) — the stream we will transform.
In a real deployment `speed` would be derived from the trajectory or fed by a
sensor; here a handful of instants makes the streaming behaviour easy to read.


In [ ]:
CID, FID, PROP = 'stream_demo', 1, 'speed'
# (re-runnable) drop any previous demo collection
S.delete(f'{HOST}/collections/{CID}')

show(S.post(f'{HOST}/collections', json={
    'id': CID, 'title': 'MF Stream demo', 'itemType': 'movingfeature',
    'crs': 'http://www.opengis.net/def/crs/EPSG/0/4326'}))

# a moving feature: a MovingPoint temporal geometry (lon/lat over time)
feature = {
  'id': FID,
  'properties': {'name': 'demo-vessel'},
  'temporalGeometry': {
    'type': 'MovingPoint',
    'datetimes': ['2026-02-26T08:00:00Z','2026-02-26T08:00:10Z','2026-02-26T08:00:20Z'],
    'coordinates': [[12.50,55.70],[12.51,55.70],[12.52,55.71]],
    'interpolation': 'Linear'}}
show(S.post(f'{HOST}/collections/{CID}/items', json=feature))

# a temporal float 'speed' in m/s
speed = {
  'name': PROP, 'type': 'TReal', 'form': 'm/s',
  'description': 'instantaneous speed over ground',
  'datetimes': ['2026-02-26T08:00:00Z','2026-02-26T08:00:05Z','2026-02-26T08:00:10Z',
                '2026-02-26T08:00:15Z','2026-02-26T08:00:20Z'],
  'values': [5.1, 7.7, 6.0, 9.3, 4.2],
  'interpolation': 'Discrete'}
show(S.post(f'{HOST}/collections/{CID}/items/{FID}/tproperties', json=speed))


## 2. Register a continuous transform

`POST …/tproperties/{name}/queries` registers a continuous query. Here we convert
speed from **m/s to knots** (`× 1.94384`) — the lifted multiplication. The response
is the OGC `cquery` link object: a `queryId`, a `status`, and the `href` of the
result stream (Server-Sent Events).

Supported operations: unary `ln, exp, log10, ceil, floor, abs, degrees, radians`;
scalar-argument `add, sub, mul, div`. (`sin/cos/tan` join the set when MEOS adds
them; everything here is an existing MEOS lifted function.)


In [ ]:
q = S.post(f'{HOST}/collections/{CID}/items/{FID}/tproperties/{PROP}/queries',
           json={'operation': 'mul', 'arg': 1.94384, 'intervalMs': 300})
show(q)
QID = q.json()['queryId']
STREAM = q.json()['href']


## 3. Consume the result stream

The `href` is a Server-Sent Events endpoint. Each `instant` event carries the
transformed value at its timestamp — speed in knots, streaming in as the source
replays. We read a few events and stop.


In [ ]:
import json as _json
def read_sse(url, n=10, timeout=30):
    with S.get(url, stream=True, timeout=timeout) as r:
        got = 0
        for line in r.iter_lines(decode_unicode=True):
            if line and line.startswith('data:'):
                ev = _json.loads(line[5:].strip())
                print(f"{ev['datetime']}  {ev['operation']}({ev['property']}) = {ev['value']:.3f} kn")
                got += 1
                if got >= n: break
read_sse(STREAM, n=10)


## 4. Lifecycle — status and stop

`GET …/queries/{queryId}` reports the live status; `DELETE` stops the query.
The lifecycle (`registered → running → stopped`) is identical across engines.


In [ ]:
show(S.get(f'{HOST}/collections/{CID}/items/{FID}/tproperties/{PROP}/queries/{QID}'))
show(S.delete(f'{HOST}/collections/{CID}/items/{FID}/tproperties/{PROP}/queries/{QID}'))
# after stopping, the query is gone
print('after stop:', S.get(f'{HOST}/collections/{CID}/items/{FID}/tproperties/{PROP}/queries/{QID}').status_code)


## What just happened

- A `tfloat` stream was transformed by a **lifted MEOS function**, in process,
  per record — no SQL, no database in the streaming path.
- Because a stream record is an **instant**, the lift is **exact**: every
  function applies pointwise without approximation.
- The whole exchange was plain HTTP + SSE against the `StreamEngine` seam. The
  same notebook drives the **Flink**, **Kafka Streams**, or **Spark Structured
  Streaming** engines once selected — they plug into the seam exactly as
  PostgreSQL, DuckDB and Spark plug into the request–response `Backend` seam.

Try other transforms: `{'operation': 'ln'}`, `{'operation': 'abs'}`,
`{'operation': 'add', 'arg': -5}`. Chaining (the streamed result feeding another
query) and windowed aggregation (`AVG`/`MAX` over tumbling windows) are the next
steps of the Part 4 surface.


## Switching the engine

Everything above ran on the default in-process `meos-local` engine. Because the
control plane is engine-neutral, the **same notebook** runs unchanged on a
cluster engine — you switch the engine where the tier starts, not in the client:

```
# in-process MEOS (default)
MFAPI_DSN=<dsn> ./mfapi

# Flink: run each continuous query as a Flink DataStream job (see flink/README.md)
MFAPI_STREAM_ENGINE=flink \
MFAPI_FLINK_CMD="java <opens> -cp <classpath> MfStreamBridgeJob" \
MFAPI_FLINK_LIBPATH=<libmeos-dir> \
MFAPI_DSN=<dsn> ./mfapi
```

The `cquery` link object, the SSE stream, and the lifecycle are identical across
engines — a Kafka Streams or Spark Structured Streaming engine plugs into the same
seam through `MFAPI_FLINK_CMD`. This mirrors the request–response tutorial, where
`MFAPI_DSN`'s scheme (`postgres://`, `duckdb:`, `spark:`) switches the database
backend with no change to the notebook.
